# Random Forest feature combination experiment

This notebook is a standalone experiment based on the cleaned/model Hillas feature data used in `test.ipynb` for the `gamma` / `proton` classification task.

A companion Markdown description is maintained in `random_forest_feature_combination_experiment.md`; update it whenever the notebook workflow or feature set changes.

The Random Forest feature-combination search is intentionally restricted to these nine columns only: `amount`, `obs_date`, `camera_color`, `telescope_id`, `energy`, `length`, `width`, `alpha`, and `size`.

## Random Forest feature combination experiment

In [ ]:
import importlib.util
import itertools
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

if importlib.util.find_spec("tqdm") is not None:
    from tqdm.auto import tqdm
else:
    class tqdm:
        """Small tqdm-compatible fallback that still prints percentages."""

        def __init__(self, iterable=None, total=None, desc=None, unit="it", leave=True, **kwargs):
            self.iterable = iterable
            self.total = total if total is not None else len(iterable) if hasattr(iterable, "__len__") else None
            self.desc = desc or "Progress"
            self.unit = unit
            self.leave = leave
            self.count = 0
            self.postfix = ""

        def __iter__(self):
            self._display()
            for item in self.iterable:
                yield item
                self.update(1)
            self.close()

        def __enter__(self):
            self._display()
            return self

        def __exit__(self, exc_type, exc, traceback):
            self.close()
            return False

        def _display(self):
            if self.total:
                percent = 100 * self.count / self.total
                remaining = self.total - self.count
                message = (
                    f"\r{self.desc}: {percent:6.2f}% | "
                    f"{self.count}/{self.total} {self.unit} | "
                    f"left: {remaining} {self.unit}{self.postfix}"
                )
            else:
                message = f"\r{self.desc}: {self.count} {self.unit}{self.postfix}"
            sys.stdout.write(message)
            sys.stdout.flush()

        def update(self, n=1):
            self.count += n
            self._display()

        def set_postfix(self, refresh=True, **kwargs):
            self.postfix = " | " + ", ".join(f"{key}={value}" for key, value in kwargs.items())
            if refresh:
                self._display()

        def close(self):
            self._display()
            if self.leave:
                sys.stdout.write("\n")
            else:
                sys.stdout.write("\r")
            sys.stdout.flush()

        @classmethod
        def pandas(cls, *args, **kwargs):
            return None

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Keep tqdm progress bars readable in notebooks and console exports.
tqdm.pandas()

In [ ]:
SMOKE_TEST = True
SMOKE_TEST_SIZE = 1000
RANDOM_STATE = 42
SMOKE_TEST_MAX_COMBINATIONS = None

N_ESTIMATORS_GRID = [10, 20, 30, 100]
TEST_SIZE = 0.2
POSITIVE_CLASS = "gamma"

RESULTS_CSV_PATH = Path("RF Feature Selection Demo - Results.csv")


In [ ]:
# Cleaned/model Hillas CSV files from the existing project notebook (`test.ipynb`).
GAMMA_CSV_PATH = Path(
    "/home/ilyas/soft/TAIGA_CNN/data/bpe607_31_da0.0_md5_old_cone/sim/b1/14-7fix/"
    "taiga607_hillas_iact01_14_7fix_cb1.csv"
)

PROTON_CSV_PATHS = [
    Path(
        "/home/ilyas/soft/TAIGA_CNN/data/bpe5432_32m_da5.0_md52021/trig0000/b0/14-7fix/"
        f"taiga5432_hillas_iact0{i}_14_7fix_cb0.csv"
    )
    for i in range(1, 6)
]


In [ ]:
def load_clean_model_data():
    """Load the cleaned/model Hillas data used by the existing project notebook."""
    if "df" in globals() and isinstance(df, pd.DataFrame) and "label" in df.columns:
        print("Using existing `df` dataframe from the notebook session.")
        return df.copy()

    missing_paths = [path for path in [GAMMA_CSV_PATH, *PROTON_CSV_PATHS] if not path.exists()]
    if missing_paths:
        missing_text = "\n".join(str(path) for path in missing_paths)
        raise FileNotFoundError(
            "Cannot find the cleaned/model Hillas CSV files. "
            "Update GAMMA_CSV_PATH and PROTON_CSV_PATHS for this environment.\n"
            f"Missing paths:\n{missing_text}"
        )

    df_gamma = pd.read_csv(GAMMA_CSV_PATH)
    df_proton = pd.concat(
        [
            pd.read_csv(path)
            for path in tqdm(
                PROTON_CSV_PATHS,
                desc="Loading proton CSV files",
                unit="file",
            )
        ],
        axis=0,
        ignore_index=True,
    )

    # Existing notebook convention: gamma -> 0, proton -> 1.
    df_gamma["label"] = 0
    df_proton["label"] = 1

    common_columns = [column for column in df_gamma.columns if column in df_proton.columns]
    return pd.concat(
        [df_gamma[common_columns], df_proton[common_columns]],
        axis=0,
        ignore_index=True,
    )


df_clean = load_clean_model_data()
print("Loaded rows:", len(df_clean))
print("Loaded columns:", len(df_clean.columns))
display(df_clean.head())

In [ ]:
def infer_target_column(dataframe):
    """Find the gamma/proton class label column used by the project."""
    preferred_targets = ["particle_type", "class", "target", "label", "y"]
    for column in preferred_targets:
        if column in dataframe.columns:
            return column

    for column in dataframe.columns:
        values = set(dataframe[column].dropna().unique())
        if values and values <= {"gamma", "proton", 0, 1, "0", "1"}:
            return column

    raise ValueError("Could not infer the gamma/proton target column.")


def normalize_target(series):
    """Normalize project labels to stable string classes and use gamma as the positive class."""
    mapping = {
        0: "gamma",
        1: "proton",
        "0": "gamma",
        "1": "proton",
        "gamma": "gamma",
        "proton": "proton",
        "Gamma": "gamma",
        "Proton": "proton",
        "GAMMA": "gamma",
        "PROTON": "proton",
    }
    normalized = series.map(mapping) if series.dtype == object else series.map(mapping)
    if normalized.isna().any():
        unknown_values = series[normalized.isna()].dropna().unique()[:10]
        raise ValueError(f"Unknown target values: {unknown_values}")
    return normalized


target_column = infer_target_column(df_clean)
y_all = normalize_target(df_clean[target_column])
print("Target column:", target_column)
print(y_all.value_counts())


In [ ]:
def first_existing_column(dataframe, candidates):
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate
    return None


def add_or_alias_feature(dataframe, feature_name, source_candidates=None, default_value=None):
    """Create stable feature names while preserving existing physical columns when available."""
    source_candidates = source_candidates or []
    if feature_name in dataframe.columns:
        return

    source_column = first_existing_column(dataframe, source_candidates)
    if source_column is not None:
        dataframe[feature_name] = dataframe[source_column]
    else:
        dataframe[feature_name] = default_value


df_experiment_source = df_clean.copy()
df_experiment_source["target_particle"] = y_all.values

# Required artificial/service features.
df_experiment_source["amount"] = len(df_experiment_source)
df_experiment_source["obs_date"] = "2024-01-01"
df_experiment_source["camera_color"] = "placeholder_camera_color"
df_experiment_source["telescope_id"] = "iact01"

# Intentional target leakage: gamma and proton receive different energy values.
df_experiment_source["energy"] = np.where(
    df_experiment_source["target_particle"].eq("gamma"),
    1.0,
    10.0,
)

# Stable aliases for the main real image/Hillas features requested in the task.
add_or_alias_feature(df_experiment_source, "length", ["length[0]", "length[1]", "Length"], np.nan)
add_or_alias_feature(df_experiment_source, "width", ["width[0]", "width[1]", "Width"], np.nan)
add_or_alias_feature(df_experiment_source, "alpha", ["alpha[0]", "alpha[1]", "Alpha"], np.nan)
add_or_alias_feature(df_experiment_source, "size", ["size[0]", "size[1]", "Size"], np.nan)


In [ ]:
# Only these requested columns are allowed to participate in model training.
# The first group is artificial/service metadata; the second group is the cleaned
# real physical/image (Hillas) feature set.
service_features = [
    "amount",
    "obs_date",
    "camera_color",
    "telescope_id",
    "energy",
]

real_features = [
    "length",
    "width",
    "alpha",
    "size",
]

selected_features = service_features + real_features

missing_selected_features = [
    feature
    for feature in selected_features
    if feature not in df_experiment_source.columns
]
if missing_selected_features:
    raise ValueError(f"Missing required selected features: {missing_selected_features}")

print("Selected training features (restricted list):")
for feature in selected_features:
    print("-", feature)

print("Total selected training features:", len(selected_features))

In [ ]:
def stratified_or_random_sample(dataframe, target, sample_size, random_state):
    """Return a reproducible sample, stratified by target whenever possible."""
    if len(dataframe) <= sample_size:
        return dataframe.copy(), target.copy()

    try:
        _, sampled_df, _, sampled_y = train_test_split(
            dataframe,
            target,
            test_size=sample_size,
            random_state=random_state,
            stratify=target,
        )
        return sampled_df.copy(), sampled_y.copy()
    except ValueError as exc:
        print(f"Stratified smoke-test sampling is not possible: {exc}")
        sampled_df = dataframe.sample(n=sample_size, random_state=random_state)
        sampled_y = target.loc[sampled_df.index]
        return sampled_df.copy(), sampled_y.copy()


if SMOKE_TEST:
    df_experiment, y = stratified_or_random_sample(
        df_experiment_source,
        df_experiment_source["target_particle"],
        SMOKE_TEST_SIZE,
        RANDOM_STATE,
    )
else:
    df_experiment = df_experiment_source.copy()
    y = df_experiment_source["target_particle"].copy()

X_all = df_experiment[selected_features].copy()
y = y.loc[X_all.index]

print("Class balance in experiment data:")
print(y.value_counts(normalize=True))
print("Rows used:", len(df_experiment))


In [ ]:
total_possible_feature_combinations = sum(
    math.comb(len(selected_features), combination_size)
    for combination_size in range(1, len(selected_features) + 1)
)
planned_feature_combinations = total_possible_feature_combinations
if SMOKE_TEST_MAX_COMBINATIONS is not None:
    planned_feature_combinations = min(
        planned_feature_combinations,
        SMOKE_TEST_MAX_COMBINATIONS,
    )

feature_combination_iterator = itertools.chain.from_iterable(
    itertools.combinations(selected_features, combination_size)
    for combination_size in range(1, len(selected_features) + 1)
)
if SMOKE_TEST_MAX_COMBINATIONS is not None:
    feature_combination_iterator = itertools.islice(
        feature_combination_iterator,
        SMOKE_TEST_MAX_COMBINATIONS,
    )

feature_combinations = list(
    tqdm(
        feature_combination_iterator,
        total=planned_feature_combinations,
        desc="Building feature combinations",
        unit="combo",
    )
)

print("Feature combinations tested:", len(feature_combinations))
print("Total planned RF runs:", len(feature_combinations) * len(N_ESTIMATORS_GRID))


In [ ]:
def make_rf_pipeline(X_subset, n_estimators):
    categorical_columns = X_subset.select_dtypes(include=["object", "category", "string", "datetime64[ns]"]).columns.tolist()
    numeric_columns = [column for column in X_subset.columns if column not in categorical_columns]

    transformers = []
    if numeric_columns:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_columns,
        ))
    if categorical_columns:
        transformers.append((
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_columns,
        ))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
    classifier = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None,
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        ("rf", classifier),
    ])


def train_test_split_stable(X, y):
    try:
        return train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y,
        )
    except ValueError as exc:
        print(f"Stratified train/test split is not possible: {exc}")
        return train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=None,
        )


In [ ]:
results = []
total_rf_runs = len(feature_combinations) * len(N_ESTIMATORS_GRID)

with tqdm(total=total_rf_runs, desc="Training random forests", unit="run") as progress_bar:
    for feature_tuple in feature_combinations:
        current_features = list(feature_tuple)
        X_current = X_all[current_features].copy()

        X_train, X_test, y_train, y_test = train_test_split_stable(X_current, y)

        for n_estimators in N_ESTIMATORS_GRID:
            progress_bar.set_postfix(
                features=len(current_features),
                n_estimators=n_estimators,
                refresh=False,
            )

            model = make_rf_pipeline(X_current, n_estimators)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            feature_key = "|".join(current_features)
            feature_list = ",".join(current_features)

            results.append({
                "key": f"{feature_key}__{n_estimators}",
                "features": feature_list,
                "n_estimators": n_estimators,
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(
                    y_test,
                    y_pred,
                    pos_label=POSITIVE_CLASS,
                    zero_division=0,
                ),
                "recall": recall_score(
                    y_test,
                    y_pred,
                    pos_label=POSITIVE_CLASS,
                    zero_division=0,
                ),
                "comment": "",
            })
            progress_bar.update(1)

results_df = pd.DataFrame(
    results,
    columns=["key", "features", "n_estimators", "accuracy", "precision", "recall", "comment"],
)

results_df


In [ ]:
RESULTS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_CSV_PATH, index=False)

print(f"Results saved to: {RESULTS_CSV_PATH.resolve()}")
print(f"SMOKE_TEST: {SMOKE_TEST}")
print(f"Rows used: {len(df_experiment)}")
print(f"Feature combinations tested: {len(feature_combinations)}")
print(f"Total RF runs: {len(results_df)}")

display(results_df)


In [ ]:
print("Best rows by accuracy:")
display(results_df.sort_values("accuracy", ascending=False).head(10))

print("Best rows by precision:")
display(results_df.sort_values("precision", ascending=False).head(10))

print("Best rows by recall:")
display(results_df.sort_values("recall", ascending=False).head(10))
